## ICE 2nd Round Statistics Question

Q1: You play a game where you randomly draw from a uniform distribution between 0 and 1. You have the option to redraw if you don't like your initial number, but you must keep the 2nd number you draw. Under optimum play, what is your expected value.

Q2: Write the general expression give a threshold 't' at which you redraw.

Q3: If you're playing against another player, assuming you're both playing optimally. What is the new threshold?

Q4: What happens when more players enter the game?

Q1:

In [3]:
t = 0.5
E_X = ((1 - t)**2)/2 + t * (1 - 0) 
print(f"Expected value: {E_X}. This assumes threshold of t: {t}")

Expected value: 0.625. This assumes threshold of t: 0.5


In [ ]:
# Empirical investigation

import random

def draw_number(threshold):

    x = random.random()
    if x <= threshold:
        return random.random()
    else:
        return x

t = 0.5 
N = 1000000
result = []
for _ in range(N):
    result.append(draw_number(t))
print(sum(result)/N)

0.6251618975200762


Q2:

In [45]:
import random

def play_turn(threshold):
    x = random.random()
    if x < threshold:
        x = random.random()
    return x

def win_rate(t1, t2, games=5000):
    """Probability that player 1 beats player 2."""
    wins = 0
    for _ in range(games):
        if play_turn(t1) > play_turn(t2):
            wins += 1
    return wins / games


def self_optimise(
    iterations=200,
    games=5000,
    step=0.05,
    decay=0.995,
):
    # Random initial thresholds
    t1 = 0.5
    t2 = 0.5

    history = []

    for _ in range(iterations):

        # ---- Player 1 best response ----
        current = win_rate(t1, t2, games)

        candidate = min(1.0, max(0.0, t1 + random.uniform(-step, step)))
        score = win_rate(candidate, t2, games)

        if score > current:
            t1 = candidate

        # ---- Player 2 best response ----
        current = 1 - win_rate(t1, t2, games)

        candidate = min(1.0, max(0.0, t2 + random.uniform(-step, step)))
        score = 1 - win_rate(t1, candidate, games)

        if score > current:
            t2 = candidate

        step *= decay
        history.append((t1, t2))

    return t1, t2, history


t1, t2, history = self_optimise(iterations=1000, games=10000)

print(f"Player 1 threshold: {t1:.4f}")
print(f"Player 2 threshold: {t2:.4f}")
print(f"P1 win rate: {win_rate(t1, t2, 100000):.4f}")

Player 1 threshold: 0.5313
Player 2 threshold: 0.6002
P1 win rate: 0.4952
